# Initiation step

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

#Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, T.StringType):
        trimmed_col = F.trim(F.col(field.name))
        
        df = df.withColumn(
            field.name, 
            F.when(trimmed_col == "", F.lit(None).cast("string"))
             .otherwise(trimmed_col)
        )

##Standarization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

## Remove Records with Missing Customer ID

In [0]:
df = df.filter(F.col("cst_id").isNotNull())

##Renaming Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity Checks of Datframe

In [0]:
df.limit(10).display()

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("workspace.silver.crm_customer")

##Sanity Checks of Silver Table

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_customer
LIMIT 10